In [ ]:
!pip install scikit-bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.9 MB/s eta 0:00:00


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist, squareform

from skbio.stats.distance import DistanceMatrix, permanova

from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

from sklearn.ensemble import IsolationForest

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score, r2_score, mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_excel("Data_vascular.xlsx")

In [ ]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,20,1.20,1.0,100,80,100,70,1
1,23,1.10,0.9,98,78,98,78,1
2,22,1.40,1.1,96,85,99,79,1
3,27,1.20,1.2,90,89,90,83,1
4,21,1.80,0.8,92,95,96,82,1
5,17,1.30,0.9,96,88,92,72,1
6,13,1.70,0.7,100,92,97,71,1
7,67,0.30,1.9,77,50,40,25,0
8,69,2.00,2.2,74,55,44,19,0
9,71,0.30,1.8,68,43,39,35,0


In [ ]:
# Shuffle original dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,71,0.30,1.8,68,43,39,35,0
1,68,0.34,1.9,75,47,35,34,0
2,20,1.20,1.0,100,80,100,70,1
3,79,0.19,1.8,70,43,41,28,0
4,17,1.30,0.9,96,88,92,72,1
5,69,2.00,2.2,74,55,44,19,0
6,22,1.40,1.1,96,85,99,79,1
7,23,1.10,0.9,98,78,98,78,1
8,70,0.35,2.1,69,59,46,22,0
9,21,1.80,0.8,92,95,96,82,1


In [ ]:
# Standardization
def standardization(x):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    return x_scaled, scaler

# Anomaly Detection

### Isolation Forest

In [ ]:
def isolation_forest(df, label_col=None, contamination=0.1):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Isolation Forest.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    # feature matrix
    X = df[feature_cols]

    # standardization (your existing function)
    X_scaled, scaler = standardization(X)

    # Isolation Forest model
    model = IsolationForest(contamination=contamination, random_state=42)

    model.fit(X_scaled)
    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/isolation_forest_model.pkl"

    joblib.dump(artifact, model_path)

    print("Model saved successfully at:", model_path)

In [ ]:
isolation_forest(df, label_col="Group")

Model saved successfully at: /content/drive/MyDrive/rasp/models/isolation_forest_model.pkl


# Predictive Modelling

## Classification

In [ ]:
# Validation
def validate_input(df, label_col):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")
    if df.empty:
        raise ValueError("Dataset is empty.")
    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataset.")
    if df[label_col].nunique() != 2:
        raise ValueError("Binary classification requires exactly 2 classes.")
    if len(df) < 10:
        raise ValueError("Dataset too small for modeling.")

### 1). Logistic Regression

#### a) Ridge

In [ ]:
def tune_ridge_logistic_classification(df, label_col, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Ridge)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
tune_ridge_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.1,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Ridge)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 

In [ ]:
# Hyperparameter tuning
tuning_results = tune_ridge_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]

Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
best_C

0.1

In [ ]:
def retrain_ridge_logistic(df, label_col, best_C):

    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=best_C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/ridge_logistic_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_ridge_logistic(df, "Group", best_C)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/ridge_logistic_model.pkl',
 'num_samples': 14,
 'num_features': 7}

#### b) Lasso

In [ ]:
def tune_lasso_logistic_classification(df, label_col, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Ridge)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
tune_lasso_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.46415888336127786,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [0.5,
    0.5,
    0.8333333333333334,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0],
   'validation_auc': [0.5,
    0.5,
    0.8333333333333334,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Ridge)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support'

In [ ]:
# Hyperparameter tuning
tuning_results = tune_lasso_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]

Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
best_C

0.46415888336127786

In [ ]:
def retrain_lasso_logistic(df, label_col, best_C):

    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=best_C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/lasso_logistic_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_lasso_logistic(df, "Group", best_C)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/lasso_logistic_model.pkl',
 'num_samples': 14,
 'num_features': 7}

#### c) ElasticNet

In [ ]:
def tune_elasticnet_logistic_classification(df, label_col, test_data_size=0.28):

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if len(X_train) < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)

    if total_samples < 50:
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        c_range = np.logspace(-3, 3, 20)

    # Elastic Net requires l1_ratio
    l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

    parameter_grid = {
        "C": c_range,
        "l1_ratio": l1_ratios
    }

    # Base Model
    base_model = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=2000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    # For frontend, show curve for best l1_ratio
    best_l1_ratio = grid.best_params_["l1_ratio"]

    filtered_results = cv_results[cv_results["param_l1_ratio"] == best_l1_ratio]

    c_values = filtered_results["param_C"].astype(float)
    train_scores = filtered_results["mean_train_score"]
    val_scores = filtered_results["mean_test_score"]

    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_l1_ratio": float(best_l1_ratio),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Elastic Net)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
# Hyperparameter tuning
tuning_results = tune_elasticnet_logistic_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]
best_l1_ratio = tuning_results["model_selection"]["best_l1_ratio"]

Fitting 3 folds for each of 50 candidates, totalling 150 fits


In [ ]:
best_C

0.1

In [ ]:
best_l1_ratio

0.1

In [ ]:
def retrain_elasticnet_logistic(df, label_col, best_C, best_l1_ratio):

    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=best_C,
        l1_ratio=best_l1_ratio,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/elasticnet_logistic_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_elasticnet_logistic(df, "Group", best_C, best_l1_ratio)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/elasticnet_logistic_model.pkl',
 'num_samples': 14,
 'num_features': 7}

## 2). SVM

In [ ]:
def tune_svm_classification(df, label_col, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV folds
    if len(X_train) < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Adaptive C range
    total_samples = len(df)

    if total_samples < 50:
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = [
        {
            "kernel": ["linear"],
            "C": c_range
        },
        {
            "kernel": ["rbf"],
            "C": c_range,
            "gamma": ["scale", 0.01, 0.1, 1]
        }
    ]

    base_model = SVC(probability=True, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, return_train_score=True, verbose=1)
    grid.fit(X_train_scaled, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_score = grid.best_score_

    # Extract CV curve (for best kernel only)
    cv_results = pd.DataFrame(grid.cv_results_)
    best_kernel = best_params["kernel"]

    filtered = cv_results[cv_results["param_kernel"] == best_kernel]

    c_vals = filtered["param_C"].astype(float)
    train_scores = filtered["mean_train_score"]
    val_scores = filtered["mean_test_score"]

    sorted_idx = np.argsort(c_vals)

    cv_curve_data = {
        "C_values": c_vals.iloc[sorted_idx].tolist(),
        "train_auc": train_scores.iloc[sorted_idx].tolist(),
        "validation_auc": val_scores.iloc[sorted_idx].tolist()
    }

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_kernel": best_params.get("kernel"),
            "best_C": float(best_params.get("C")),
            "best_gamma": best_params.get("gamma"),
            "best_cv_auc": float(best_cv_score) if best_cv_score else None,
            "c_selection_curve": cv_curve_data
        },
        "model_results": {
            "model_type": "SVM",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            }
        }
    }

In [ ]:
tune_svm_classification(df, label_col="Group", test_data_size=0.28)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_kernel': 'linear',
  'best_C': 0.1,
  'best_gamma': None,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'SVM',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-sco

In [ ]:
# Hyperparameter tuning
tuning_results = tune_svm_classification(df=df, label_col="Group", test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]
best_kernel = tuning_results["model_selection"]["best_kernel"]
best_gamma = tuning_results["model_selection"]["best_gamma"]

Fitting 3 folds for each of 50 candidates, totalling 150 fits


In [ ]:
best_C

0.1

In [ ]:
best_kernel

'linear'

In [ ]:
best_gamma

In [ ]:
def retrain_svm(df, label_col, best_C, best_kernel, best_gamma):
    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Kernel-specific model creation
    if best_kernel == "linear":
        model = SVC(kernel=best_kernel, C=best_C, probability=True, random_state=42)
    else:
        model = SVC(kernel=best_kernel, C=best_C, gamma=best_gamma, probability=True, random_state=42)

    model.fit(X_scaled, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/svm_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_svm(df, "Group", best_C, best_kernel, best_gamma)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/svm_model.pkl',
 'num_samples': 14,
 'num_features': 7}

## 3) PCA Analysis

In [ ]:
def pca_analysis_json(df, label_col, variance_threshold=0.95):
    # Select Numeric Features
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols]

    # Standardize Data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA()
    pca.fit(X_scaled)

    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    # Convert to percentage
    explained_variance_pct = explained_variance * 100
    cumulative_variance_pct = cumulative_variance * 100
    threshold_pct = variance_threshold * 100

    # Compute Threshold Components
    components_needed = int(np.argmax(cumulative_variance >= variance_threshold) + 1)

    # PCA Loadings
    loadings_df = pd.DataFrame(
        pca.components_.T,
        columns=[f"PC{i+1}" for i in range(len(feature_cols))],
        index=feature_cols
    )

    # Final JSON Output
    return {
        "pca_summary": {
            "total_samples": int(len(df)),
            "total_features": int(len(feature_cols)),
            "variance_threshold_percent": float(threshold_pct),
            "components_needed_for_threshold": components_needed
        },
        "scree_plot_data": {
            "components": list(range(1, len(explained_variance) + 1)),
            "explained_variance_percent": explained_variance_pct.tolist(),
            "cumulative_variance_percent": cumulative_variance_pct.tolist(),
            "threshold_line_percent": float(threshold_pct)
        },
        "pca_loadings": loadings_df.to_dict()
    }

In [ ]:
pca_analysis_json(df, label_col="Group", variance_threshold=0.95)

{'pca_summary': {'total_samples': 14,
  'total_features': 7,
  'variance_threshold_percent': 95.0,
  'components_needed_for_threshold': 2},
 'scree_plot_data': {'components': [1, 2, 3, 4, 5, 6, 7],
  'explained_variance_percent': [88.55949128638862,
   7.553270394983713,
   1.6625140169716528,
   0.9331628222968442,
   0.6928077714880654,
   0.4628311334017339,
   0.13592257446936923],
  'cumulative_variance_percent': [88.55949128638862,
   96.11276168137233,
   97.77527569834399,
   98.70843852064083,
   99.4012462921289,
   99.86407742553062,
   100.0],
  'threshold_line_percent': 95.0},
 'pca_loadings': {'PC1': {'Permeability': 0.39899903565752076,
   'LDL_uptake': -0.30119667678031936,
   'Total_ROS': 0.3848461690720713,
   'Vascular_Marker': -0.3812643884023061,
   'Cell_Signalling': -0.38996626050911803,
   'Tube_formation': -0.39600035243844306,
   'In_vivo_recovery': -0.3843452357722018},
  'PC2': {'Permeability': 0.06412796620344198,
   'LDL_uptake': 0.9037985307994665,
   'To

In [ ]:
# PCA Analysis
pca_results = pca_analysis_json(df=df, label_col="Group", variance_threshold=0.95)

total_components = pca_results["pca_summary"]["components_needed_for_threshold"]

In [ ]:
total_components

2

### a) Logistic Regression

#### i). Ridge

In [ ]:
def tune_pca_ridge_logistic_classification(df, label_col, n_components, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # PCA
    pca = PCA(n_components=n_components)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_pca, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_pca)
    y_prob = best_model.predict_proba(X_test_pca)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coefficients = best_model.coef_[0]

    component_names = [f"PC{i+1}" for i in range(len(coefficients))]

    component_importance = dict(
        sorted(
            zip(component_names, coefficients),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    )

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "component_importance": component_importance
        }
    }

In [ ]:
tune_pca_ridge_logistic_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.1,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 4.0},
   'weighted avg': {'precision': 1.0,
   

In [ ]:
# Hyperparameter tuning
tuning_results = tune_pca_ridge_logistic_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]

Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
best_C

0.1

In [ ]:
def retrain_pca_ridge_logistic(df, label_col, n_components, best_C):
    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    model = LogisticRegression(penalty="l2", C=best_C, solver="lbfgs", max_iter=5000, random_state=42)
    model.fit(X_pca, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "pca": pca,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/ridge_logistic_pca_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_pca_ridge_logistic(df, "Group", total_components, best_C)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/ridge_logistic_pca_model.pkl',
 'num_samples': 14,
 'num_features': 7}

#### ii). Lasso

In [ ]:
def tune_pca_lasso_logistic_classification(df, label_col, n_components, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # PCA
    pca = PCA(n_components=n_components)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_pca, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_pca)
    y_prob = best_model.predict_proba(X_test_pca)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coefficients = best_model.coef_[0]

    component_names = [f"PC{i+1}" for i in range(len(coefficients))]

    component_importance = dict(
        sorted(
            zip(component_names, coefficients),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    )

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "component_importance": component_importance
        }
    }

In [ ]:
tune_pca_lasso_logistic_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.16681005372000587,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 4.0},
   'weighted avg': {'prec

In [ ]:
# Hyperparameter tuning
tuning_results = tune_pca_lasso_logistic_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

best_C = tuning_results["model_selection"]["best_C"]

Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
best_C

0.16681005372000587

In [ ]:
def retrain_pca_lasso_logistic(df, label_col, n_components, best_C):
    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    model = LogisticRegression(penalty="l1", C=best_C, solver="liblinear", max_iter=5000, random_state=42)
    model.fit(X_pca, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "pca": pca,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/lasso_logistic_pca_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_pca_lasso_logistic(df, "Group", total_components, best_C)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/lasso_logistic_pca_model.pkl',
 'num_samples': 14,
 'num_features': 7}

#### iii) Elastic Net

In [ ]:
def tune_pca_elasticnet_logistic_classification(df, label_col, n_components, test_data_size=0.28):

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # PCA
    pca = PCA(n_components=n_components)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    if len(X_train) < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)

    if total_samples < 50:
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        c_range = np.logspace(-3, 3, 20)

    # Elastic Net requires l1_ratio
    l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

    parameter_grid = {
        "C": c_range,
        "l1_ratio": l1_ratios
    }

    # Base Model
    base_model = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=2000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_pca, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    # For frontend, show curve for best l1_ratio
    best_l1_ratio = grid.best_params_["l1_ratio"]

    filtered_results = cv_results[cv_results["param_l1_ratio"] == best_l1_ratio]

    c_values = filtered_results["param_C"].astype(float)
    train_scores = filtered_results["mean_train_score"]
    val_scores = filtered_results["mean_test_score"]

    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_pca)
    y_prob = best_model.predict_proba(X_test_pca)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Component Importance
    coefficients = best_model.coef_[0]
    component_names = [f"PC{i+1}" for i in range(len(coefficients))]

    component_importance = dict(
        sorted(
            zip(component_names, coefficients),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    )

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_l1_ratio": float(best_l1_ratio),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Elastic Net)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "component_importance": component_importance
        }
    }

In [ ]:
tune_pca_elasticnet_logistic_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.1,
  'best_l1_ratio': 0.1,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Elastic Net)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f

In [ ]:
# Hyperparameter tuning
tuning_results = tune_pca_elasticnet_logistic_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

best_C_pca = tuning_results["model_selection"]["best_C"]
best_l1_ratio_pca = tuning_results["model_selection"]["best_l1_ratio"]

Fitting 3 folds for each of 50 candidates, totalling 150 fits


In [ ]:
best_C_pca

0.1

In [ ]:
best_l1_ratio_pca

0.1

In [ ]:
def retrain_pca_elasticnet_logistic(df, label_col, n_components, best_C, best_l1_ratio):
    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    model = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=best_C,
        l1_ratio=best_l1_ratio,
        max_iter=5000,
        random_state=42
    )

    model.fit(X_pca, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "pca": pca,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/elasticnet_logistic_pca_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
retrain_pca_elasticnet_logistic(df, label_col="Group", n_components=total_components, best_C=best_C_pca, best_l1_ratio=best_l1_ratio_pca)

{'model_path': '/content/drive/MyDrive/rasp/models/elasticnet_logistic_pca_model.pkl',
 'num_samples': 14,
 'num_features': 7}

### b) SVM

In [ ]:
def tune_pca_svm_classification(df, label_col, n_components, test_data_size=0.28):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=n_components)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Adaptive CV folds
    if len(X_train) < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Adaptive C range
    total_samples = len(df)

    if total_samples < 50:
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = [
        {
            "kernel": ["linear"],
            "C": c_range
        },
        {
            "kernel": ["rbf"],
            "C": c_range,
            "gamma": ["scale", 0.01, 0.1, 1]
        }
    ]

    base_model = SVC(probability=True, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, return_train_score=True, verbose=1)
    grid.fit(X_train_pca, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_score = grid.best_score_

    # Extract CV curve (for best kernel only)
    cv_results = pd.DataFrame(grid.cv_results_)
    best_kernel = best_params["kernel"]

    filtered = cv_results[cv_results["param_kernel"] == best_kernel]

    c_vals = filtered["param_C"].astype(float)
    train_scores = filtered["mean_train_score"]
    val_scores = filtered["mean_test_score"]

    sorted_idx = np.argsort(c_vals)

    cv_curve_data = {
        "C_values": c_vals.iloc[sorted_idx].tolist(),
        "train_auc": train_scores.iloc[sorted_idx].tolist(),
        "validation_auc": val_scores.iloc[sorted_idx].tolist()
    }

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_pca)
    y_prob = best_model.predict_proba(X_test_pca)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_kernel": best_params.get("kernel"),
            "best_C": float(best_params.get("C")),
            "best_gamma": best_params.get("gamma"),
            "best_cv_auc": float(best_cv_score) if best_cv_score else None,
            "c_selection_curve": cv_curve_data
        },
        "model_results": {
            "model_type": "SVM",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            }
        }
    }

In [ ]:
tune_pca_svm_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_kernel': 'linear',
  'best_C': 0.1,
  'best_gamma': None,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'SVM',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-sco

In [ ]:
# Hyperparameter tuning
tuning_results = tune_pca_svm_classification(df, label_col="Group", n_components=total_components, test_data_size=0.28)

best_C_pca = tuning_results["model_selection"]["best_C"]
best_kernel_pca = tuning_results["model_selection"]["best_kernel"]
best_gamma_pca = tuning_results["model_selection"]["best_gamma"]

Fitting 3 folds for each of 50 candidates, totalling 150 fits


In [ ]:
def retrain_pca_svm(df, label_col, n_components, best_C, best_kernel, best_gamma):
    validate_input(df, label_col)

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]
    y = df[label_col]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Kernel-specific model creation
    if best_kernel == "linear":
        model = SVC(kernel=best_kernel, C=best_C, probability=True, random_state=42)
    else:
        model = SVC(kernel=best_kernel, C=best_C, gamma=best_gamma, probability=True, random_state=42)

    model.fit(X_pca, y)

    artifact = {
        "model": model,
        "scaler": scaler,
        "pca": pca,
        "features": feature_cols
    }

    model_path = "/content/drive/MyDrive/rasp/models/svm_pca_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "model_path": model_path,
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols))
    }

In [ ]:
# Retrain final model
train_results = retrain_pca_svm(df, "Group", total_components, best_C_pca, best_kernel_pca, best_gamma_pca)

In [ ]:
train_results

{'model_path': '/content/drive/MyDrive/rasp/models/svm_pca_model.pkl',
 'num_samples': 14,
 'num_features': 7}

# Clustering Analysis

## 1) K-Means Clustering

In [ ]:
def auto_select_k(df, label_col=None, max_clusters=10):

    # Select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    X = df[feature_cols].copy()

    # Missing value check
    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found. Handle NaNs before clustering")

    # Scale
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(X)

    n_samples = x_scaled.shape[0]

    if n_samples < 4:
        raise ValueError("Dataset too small for clustering")

    max_k = min(max_clusters, n_samples // 2)

    ks = []
    silhouettes = []

    for k in range(2, max_k + 1):
        kmeans = KMeans(n_clusters=k, init="k-means++", n_init="auto", random_state=42)

        labels = kmeans.fit_predict(x_scaled)

        # Handle edge case (single cluster)
        if len(set(labels)) > 1:
            score = silhouette_score(x_scaled, labels)
        else:
            score = -1

        ks.append(k)
        silhouettes.append(score)

    best_k = ks[np.argmax(silhouettes)]

    return {
        "k_values": ks,
        "silhouette": silhouettes,
        "best_k": best_k,
    }

In [ ]:
def train_kmeans(df, label_col=None, max_clusters=10):

    # Select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in df.columns:
        if label_col in feature_cols:
            feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    X = df[feature_cols].copy()

    # Validation
    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found. Handle NaNs before training")

    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Auto k
    result = auto_select_k(df, label_col, max_clusters)

    best_k = result["best_k"]
    ks = result["k_values"]
    silhouettes = result["silhouette"]

    # Train final model
    model = KMeans(n_clusters=best_k, init="k-means++", n_init="auto", random_state=42)

    model.fit(X_scaled)

    # Save artifact
    artifact = {
        "model": model,
        "scaler": scaler,
        "features": feature_cols,
        "k": best_k,
    }

    model_path = "/content/drive/MyDrive/rasp/models/k_means_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "best_k": best_k,
        "features_used": feature_cols,
        "silhouette_scores": dict(zip(ks, silhouettes))
    }

In [ ]:
train_kmeans(df, label_col="Group", max_clusters=10)

{'best_k': 2,
 'features_used': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'silhouette_scores': {2: np.float64(0.7546440860425349),
  3: np.float64(0.6915017165248429),
  4: np.float64(0.4646839635532535),
  5: np.float64(0.2951237781064383),
  6: np.float64(0.38578664458181583),
  7: np.float64(0.36057454069780354)}}

## 2). Gaussian Mixture Model (GMM)

In [ ]:
def auto_select_gmm_k(df, label_col=None, max_clusters=10):
    # Select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    X = df[feature_cols].copy()

    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found")

    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_samples = X_scaled.shape[0]

    if n_samples < 4:
        raise ValueError("Dataset too small")

    max_k = min(max_clusters, n_samples // 2)

    ks, bic_scores = [], []

    for k in range(2, max_k + 1):
        gmm = GaussianMixture(n_components=k, covariance_type="full", n_init=10, random_state=42)

        gmm.fit(X_scaled)

        ks.append(k)
        bic_scores.append(gmm.bic(X_scaled))

    best_k = ks[np.argmin(bic_scores)]

    return {
        "k_values": ks,
        "bic": bic_scores,
        "best_k": best_k,
        "features": feature_cols
    }

In [ ]:
def train_gmm(df, label_col=None, max_clusters=10):
    # Select features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    X = df[feature_cols].copy()

    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found")

    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Auto k
    result = auto_select_gmm_k(df, label_col, max_clusters)
    best_k = result["best_k"]

    # Train GMM
    model = GaussianMixture(n_components=best_k, covariance_type="full", n_init=20, random_state=42)

    model.fit(X_scaled)

    # PCA (for consistent visualization)
    pca = PCA(n_components=2)
    pca.fit(X_scaled)

    # Save artifact
    artifact = {
        "model": model,
        "scaler": scaler,
        "pca": pca,
        "features": feature_cols,
        "k": best_k
    }

    model_path = "/content/drive/MyDrive/rasp/models/GMM_model.pkl"

    joblib.dump(artifact, model_path)

    return {
        "best_k": best_k,
        "features_used": feature_cols,
        "bic_scores": dict(zip(result["k_values"], result["bic"]))
    }

In [ ]:
train_gmm(df, label_col="Group", max_clusters=10)

{'best_k': 6,
 'features_used': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'bic_scores': {2: np.float64(-55.04696300161228),
  3: np.float64(-142.6215820241665),
  4: np.float64(-179.45212117350167),
  5: np.float64(-253.479827755183),
  6: np.float64(-286.23326157695647),
  7: np.float64(-234.72851286130538)}}

## 4). DBSCAN